# Heatmap Regional por Ano

Dois heatmaps por variável (Casos, Mortalidade, Mamografia), um
EMBAIXO do outro -- eixo Y = Regional de Saúde, eixo X = Ano, um
colorido por QUANTIDADE (valor absoluto), outro por TAXA por 100 mil
habitantes, cada um com o seu próprio valor escrito dentro da célula
(a quantidade no heatmap de quantidade, a taxa no de taxa -- nunca as
duas no mesmo heatmap). Hospitais fica de fora (só tem 1 ano, não faz
sentido como Regional × Ano).

Cada Regional já vem com sua própria TAXA calculada por
`calculo_taxa.ipynb` (razão das somas de QUANTIDADE/POPULACAO daquela
Regional) -- este notebook só pivota o CSV de taxa para o formato
Regional × Ano, sem nenhuma agregação adicional entre Regionais.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

from caminho_utilitarios import preparar_sys_path
preparar_sys_path()


PosixPath('/home/anna/tcc/Seleção e Transformação/População')

In [2]:
from utilitarios_comuns import (
    COLUNA_REGIONAL,
    PASTA_POPULACAO,
    ESPECIFICACOES_VARIAVEIS,
    registrar_log,
    selecionar_colunas_de_valor,
    verificar_anos_unicos,
)
from utilitarios_graficos_populacionais import (
    carregar_taxa_por_regional,
    carregar_mapa_regional_original,
    nomes_regional_originais,
    preparar_simples_por_regional_ano,
    montar_par_heatmaps_regional_ano,
    salvar_figura,
)

### Configuração

In [3]:
PASTA_TAXAS = PASTA_POPULACAO / "Taxa"
PASTA_SAIDA_GRAFICOS = Path.cwd() / "Gráficos" / "Heatmap Regional"

SALVAR_FIGURAS = False

MAPA_REGIONAL_ORIGINAL = carregar_mapa_regional_original()

# Intervalo de anos exibido; None usa o período completo da variável
ANO_INICIAL_EXIBIDO = 2013
ANO_FINAL_EXIBIDO = 2023

# Tamanho da figura
LARGURA_FIGURA = 1400
ALTURA_FIGURA = 1700

# Fontes dos eixos e dos valores dentro da célula
TAMANHO_FONTE_TITULO_EIXO_HEATMAP = 20
TAMANHO_FONTE_TICK_EIXO_HEATMAP = 13
TAMANHO_FONTE_VALOR_HEATMAP = 12

# Escreve o valor dentro de cada célula
MOSTRAR_VALORES_HEATMAP = True

# Espaçamento vertical entre os dois heatmaps
ESPACAMENTO_VERTICAL_SUBPLOTS = 0.12
MARGEM_FIGURA = dict(l=100, r=150, t=110, b=70)

# Escala de cor dos heatmaps, do mais claro ao mais escuro
ESCALA_COR = [
    "#F5E6E6",
    "#FEB6B6",
    "#D16E6E",
    "#B03D3D",
    "#DD0A0A",
    "#760303",
]

# NOME_EXIBICAO é opcional; sem ele, o nome exibido é a própria chave da variável
CONFIGURACAO_VISUAL_VARIAVEIS = {
    "Casos": {
        "ESCALA_COR_QUANTIDADE": ESCALA_COR, "ESCALA_COR_TAXA": ESCALA_COR,
        "TITULO_BARRA_COR_QUANTIDADE": "Casos - Valor absoluto", "TITULO_BARRA_COR_TAXA": "Taxa/100k hab.",
    },
    "Mortalidade": {
        "NOME_EXIBICAO": "Mortes",
        "ESCALA_COR_QUANTIDADE": ESCALA_COR, "ESCALA_COR_TAXA": ESCALA_COR,
        "TITULO_BARRA_COR_QUANTIDADE": "Mortes - Valor absoluto", "TITULO_BARRA_COR_TAXA": "Taxa/100k hab.",
    },
    "Mamografia": {
        "NOME_EXIBICAO": "Mamografias",
        "ESCALA_COR_QUANTIDADE": ESCALA_COR, "ESCALA_COR_TAXA": ESCALA_COR,
        "TITULO_BARRA_COR_QUANTIDADE": "Mamografias - Valor absoluto", "TITULO_BARRA_COR_TAXA": "Taxa/100k hab.",
    },
}

LISTA_VARIAVEIS = [
    {
        "NOME": nome,
        "NOME_EXIBICAO": nome,
        "CAMINHO_TAXA": PASTA_TAXAS / ESPECIFICACOES_VARIAVEIS[nome]["NOME_ARQUIVO_SAIDA_TAXA"],
        "ANO_INICIAL": ANO_INICIAL_EXIBIDO or ESPECIFICACOES_VARIAVEIS[nome]["ANO_INICIAL"],
        "ANO_FINAL": ANO_FINAL_EXIBIDO or ESPECIFICACOES_VARIAVEIS[nome]["ANO_FINAL"],
        **config_visual,
    }
    for nome, config_visual in CONFIGURACAO_VISUAL_VARIAVEIS.items()
]

### Heatmaps

In [4]:
for configuracao in LISTA_VARIAVEIS:
    nome_da_variavel = configuracao["NOME"]
    nome_exibicao = configuracao["NOME_EXIBICAO"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_taxa_regional = carregar_taxa_por_regional(configuracao["CAMINHO_TAXA"])

    colunas_de_valor = selecionar_colunas_de_valor(
        tabela_taxa_regional, configuracao["ANO_INICIAL"], configuracao["ANO_FINAL"],
        colunas_a_ignorar=(COLUNA_REGIONAL,), somente_total=True,
    )
    verificar_anos_unicos(colunas_de_valor, nome_da_variavel=nome_da_variavel)

    tabela_longa = preparar_simples_por_regional_ano(tabela_taxa_regional, colunas_de_valor)

    tabela_longa[COLUNA_REGIONAL] = nomes_regional_originais(
        tabela_longa[COLUNA_REGIONAL], MAPA_REGIONAL_ORIGINAL
    )

    tabela_pivotada_taxa = tabela_longa.pivot(index=COLUNA_REGIONAL, columns="ANO", values="TAXA")
    tabela_pivotada_quantidade = tabela_longa.pivot(index=COLUNA_REGIONAL, columns="ANO", values="QUANTIDADE")

    figura = montar_par_heatmaps_regional_ano(
        tabela_pivotada_quantidade=tabela_pivotada_quantidade,
        tabela_pivotada_taxa=tabela_pivotada_taxa,
        titulo="",
        escala_cor_quantidade=configuracao["ESCALA_COR_QUANTIDADE"],
        escala_cor_taxa=configuracao["ESCALA_COR_TAXA"],
        titulo_barra_cor_quantidade=configuracao["TITULO_BARRA_COR_QUANTIDADE"],
        titulo_barra_cor_taxa=configuracao["TITULO_BARRA_COR_TAXA"],
        titulo_subplot_quantidade="Valor absoluto",
        largura=LARGURA_FIGURA, altura=ALTURA_FIGURA,
        tamanho_fonte_titulo_eixo=TAMANHO_FONTE_TITULO_EIXO_HEATMAP,
        tamanho_fonte_tick_eixo=TAMANHO_FONTE_TICK_EIXO_HEATMAP,
        espacamento_vertical=ESPACAMENTO_VERTICAL_SUBPLOTS,
        margem=MARGEM_FIGURA,
        mostrar_valores=MOSTRAR_VALORES_HEATMAP,
        tamanho_fonte_valor=TAMANHO_FONTE_VALOR_HEATMAP,
    )
    figura.show()

    if SALVAR_FIGURAS:
        salvar_figura(figura, f"heatmap_{nome_exibicao.lower().replace(' ', '_')}", PASTA_SAIDA_GRAFICOS)

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} =====\n")
    

[LOG] ===== INÍCIO: CASOS =====


[LOG] ===== FIM: CASOS =====

[LOG] ===== INÍCIO: MORTALIDADE =====


[LOG] ===== FIM: MORTALIDADE =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====


[LOG] ===== FIM: MAMOGRAFIA =====

